# Data cleaning with pandas

**Learning Objectives:**
- Understand the importance of data cleaning in the data analysis pipeline
- Perform initial data inspection to identify quality issues
- Handle inconsistent and irrelevant data values
- Convert columns to appropriate data types
- Detect and handle missing values using different strategies
- Identify and remove duplicate rows
- Detect and handle outliers in numerical data

**Dataset:** We'll use the Titanic dataset, which contains passenger information from the famous 1912 voyage.

---

## Introduction

📚 **Concept: What is data cleaning?**

Data cleaning is the process of identifying and correcting (or removing) errors and inconsistencies in data to improve its quality. Real-world data is often "messy" and requires cleaning before analysis.

**Common data quality issues:**
- Inconsistent formatting (e.g., "$10" vs "10")
- Wrong data types (e.g., numbers stored as strings)
- Missing values (e.g., empty cells, "N/A", "Unknown")
- Duplicate records
- Outliers and data entry errors

💡 **Tip:** Data cleaning often takes 50-80% of a data scientist's time. It's not glamorous, but it's essential for accurate analysis!

---

## Recommended workflow: 6 steps for data cleaning

1. **First inspection** - Get an overview of your data
2. **Handle inconsistent or irrelevant data** - Fix formatting issues
3. **Type conversion** - Ensure columns have appropriate data types
4. **Missing values** - Decide whether to drop or impute
5. **Duplicates** - Identify and remove duplicate records
6. **Outliers** - Detect and handle extreme values

⚠️ **Warning:** Always work on a copy of your data during cleaning, so you can return to the original if needed!

<!-- INSTRUCTOR: Emphasize that this is a recommended workflow, not strict rules. Real projects may require iteration between steps. -->

## 1. First inspection

📚 **Concept:** Before cleaning data, we need to understand what we're working with. The `.info()` method gives us a quick overview of our dataset's structure and potential issues.

In [ ]:
import pandas as pd

# Load the Titanic dataset
titanic = pd.read_csv("titanic_imp.csv")

In [ ]:
# Get overview of data structure
titanic.info()

✅ **Key observations:**
- `Fare` is stored as object (string) instead of numeric - needs fixing!
- `Age` is stored as object instead of numeric - needs fixing!
- `Survived` is stored as object instead of integer - needs fixing!
- We have missing values in `Age` (758 non-null out of 894), `Emb` (892/894), and `Deck` (203/894)
- The `Deck` column is mostly missing (only 23% complete) - we may want to drop this column

<!-- INSTRUCTOR: Point out that .info() shows both data types and null counts, making it perfect for initial inspection. -->

In [ ]:
# Preview the first few rows
titanic.head()

---

## 2. Handle inconsistent and irrelevant data

📚 **Concept:** Inconsistent data refers to values that don't follow a standard format. For example, the `Fare` column includes dollar signs ("$7.25"), making it a string instead of a number.

<!-- INSTRUCTOR: Emphasize that we need to check for these issues column by column. -->

### 2.1 Cleaning the Fare column

The `Fare` column contains dollar signs that prevent it from being numeric. We'll remove them using `.str.replace()`.

In [ ]:
# Test removing the dollar sign (without saving changes yet)
titanic["Fare"].str.replace("$", "")

In [ ]:
# Apply the change to the dataframe
titanic["Fare"] = titanic["Fare"].str.replace("$", "")

# Verify it worked
titanic["Fare"].head()

In [ ]:
# Rename column to preserve information that currency is dollars
titanic = titanic.rename(columns={"Fare": "Fare ($)"})

💡 **Tip:** We use the reassignment pattern `df = df.rename()` instead of `inplace=True`. This is the modern pandas best practice because it's clearer and safer.

<!-- INSTRUCTOR: Explain why inplace=True is discouraged: it can cause issues with method chaining, doesn't work in all contexts, and makes code harder to reason about. -->

### 2.2 Finding other inconsistencies

Let's use `.describe()` to check text columns for unexpected values.

In [ ]:
# Check all object (text) columns for inconsistencies
titanic.describe(include="O")

⚠️ **Warning:** The `Survived` column has 4 unique values, but we'd expect only 2 (survived or didn't survive). Let's investigate!

In [ ]:
# Check what unique values exist in Survived column
titanic["Survived"].unique()

📚 **Concept:** This column has inconsistent encoding! Some values are numbers ("0", "1") and others are text ("yes", "no"). We need to standardize them.

<!-- INSTRUCTOR: This is a common issue when data comes from multiple sources or is manually entered. -->

In [ ]:
# Replace "yes" and "no" with 1 and 0
titanic["Survived"] = titanic["Survived"].replace(
    to_replace=["yes", "no"], 
    value=[1, 0]
)

# Verify the fix
titanic["Survived"].unique()

---

## 3. Type conversion

📚 **Concept:** Each column should have the appropriate data type for its content. Numbers should be numeric types (int, float), not strings (object).

**Why this matters:**
- Mathematical operations only work on numeric types
- Memory efficiency - numbers stored as strings take more space
- Prevent errors in analysis

<!-- INSTRUCTOR: Demonstrate what happens if you try to calculate mean() on an object column. -->

### 3.1 Converting the Survived column

In [ ]:
# Test conversion to integer (without saving yet)
titanic["Survived"].astype("int")

In [ ]:
# Check the distribution after conversion
titanic["Survived"].astype("int").value_counts()

In [ ]:
# Convert Survived column to integer
titanic["Survived"] = titanic["Survived"].astype("int")

### 3.2 Converting the Fare column

In [ ]:
# Convert Fare ($) to numeric (float)
titanic["Fare ($)"] = titanic["Fare ($)"].astype("float")

In [ ]:
# Check our progress
titanic.info()

### 3.3 The Age column problem

⚠️ **Warning:** We still need to convert `Age`, but it's currently stored as object. Let's see why...

In [ ]:
# Check the unique values in Age (including NaN)
titanic["Age"].value_counts(dropna=False)

📚 **Concept:** The `Age` column contains the string "Missing Data" instead of proper null values (`NaN`). This prevents conversion to numeric type. We need to replace "Missing Data" with `np.nan` first.

<!-- INSTRUCTOR: Explain that different data sources use different conventions for missing data: "N/A", "Unknown", "Missing", empty strings, etc. -->

In [ ]:
import numpy as np

# np.nan is pandas' standard way to represent missing numeric data
np.nan

In [ ]:
# Replace "Missing Data" with proper NaN values
titanic["Age"] = titanic["Age"].replace(to_replace="Missing Data", value=np.nan)

In [ ]:
# Now we can convert Age to float
titanic["Age"] = titanic["Age"].astype("float")

# Success! Age is now numeric
titanic["Age"].dtype

---

## 4. Missing values

📚 **Concept:** Missing data is one of the most common data quality issues. We have two main strategies:

1. **Drop** - Remove rows or columns with missing data
2. **Impute** - Fill missing values with estimates (mean, median, mode, etc.)

**Decision framework:**
- Drop when: Missing data is < 5% of total, or the column is not important
- Impute when: You have enough data to make reasonable estimates

<!-- INSTRUCTOR: Emphasize that there's no perfect answer - it depends on your specific use case and how much data you can afford to lose. -->

### 4.1 Dropping missing values with `dropna()`

Let's first see what would happen if we dropped all rows with any missing value.

In [ ]:
# Drop all rows with ANY missing value
titanic.dropna()

# Only 182 rows left out of 894! We'd lose 80% of our data!

⚠️ **Warning:** Dropping all rows with any missing value would remove 80% of our data! That's too much.

💡 **Tip:** Use `thresh` parameter to be more selective. `thresh=8` means "keep rows with at least 8 non-null values" (out of 9 columns).

In [ ]:
# Drop rows with 2 or more missing values
titanic.dropna(thresh=8)

# This keeps 736 rows - better, but still loses data

**Our strategy:** Instead of dropping rows, we'll:
1. Drop the `Deck` column (77% missing - too much to impute reliably)
2. Impute the remaining missing values in `Age` and `Emb`

<!-- INSTRUCTOR: Explain that this is a judgment call based on domain knowledge and analysis goals. -->

In [ ]:
# Check current missing values
titanic.info()

In [ ]:
# Drop the Deck column (too many missing values)
titanic = titanic.drop(columns="Deck")

# Verify it's gone
titanic.info()

### 4.2 Imputing missing values with `fillna()`

📚 **Concept:** Imputation means replacing missing values with estimated values. Common strategies:

- **Mean**: Good for numeric data without outliers
- **Median**: Better for numeric data with outliers (more robust)
- **Mode**: Best for categorical data (most frequent value)
- **Forward/backward fill**: Good for time series data

<!-- INSTRUCTOR: Discuss when each method is appropriate. Mean is affected by outliers, median is not. -->

### 4.3 Imputing Age with the mean

In [ ]:
# Calculate the mean age
titanic["Age"].mean()

In [ ]:
# Round to 1 decimal place for cleaner data
mean_age = round(titanic["Age"].mean(), 1)
mean_age

In [ ]:
# Fill missing Age values with the mean
titanic["Age"] = titanic["Age"].fillna(mean_age)

# Check that all values are now filled
titanic.info()

✅ **Key Takeaway:** Notice that `Age` now has 894 non-null entries (was 717 before). All missing values have been filled!

### 4.4 Imputing Emb with the mode

The `Emb` (embarkation port) column is categorical, so we should use the mode (most common value).

In [ ]:
# Check the Emb column
titanic["Emb"].value_counts()

In [ ]:
# Find the most common value (mode)
titanic["Emb"].mode()

In [ ]:
# Fill missing values with the mode
# .mode() returns a Series, so we use [0] to get the first (and only) value
titanic["Emb"] = titanic["Emb"].fillna(titanic["Emb"].mode()[0])

# Verify all values are filled
titanic.info()

✅ **Key Takeaway:** All columns now have 894 non-null values - no more missing data!

### ✅ Quick Check

Before proceeding, can you:
- [ ] Explain the difference between dropping and imputing missing values?
- [ ] Decide when to use mean vs. median vs. mode for imputation?
- [ ] Use `.fillna()` to fill missing values?

<!-- INSTRUCTOR: This is a good stopping point for discussion. -->

---

## 5. Duplicates

📚 **Concept:** Duplicate rows can skew analysis results by counting the same observation multiple times. They often result from:
- Data entry errors
- Merging datasets incorrectly
- System glitches

<!-- INSTRUCTOR: Emphasize that duplicates are harmful because they give more weight to certain observations in statistical analysis. -->

In [ ]:
# Check for duplicate rows
titanic.duplicated()

In [ ]:
# View the duplicate rows
titanic.loc[titanic.duplicated()]

⚠️ **Warning:** We found 114 duplicate rows! However, these might not be true duplicates - they could be different passengers with identical characteristics (same age, class, fare, etc.).

💡 **Tip:** In a real analysis, you'd want a unique identifier (like a passenger ID) to definitively identify duplicates. Since we don't have one here, we'll assume these are likely duplicates and remove them.

<!-- INSTRUCTOR: Discuss the importance of unique identifiers in real-world datasets. -->

In [ ]:
# Remove duplicate rows
titanic = titanic.drop_duplicates()

# Verify removal
print(f"Rows after removing duplicates: {len(titanic)}")

In [ ]:
# Reset index after removing rows
titanic = titanic.reset_index(drop=True)

titanic.head()

💡 **Tip:** After removing rows, it's good practice to reset the index so it's continuous (0, 1, 2, ...). The `drop=True` parameter prevents the old index from being saved as a new column.

📚 **Modern pandas:** Notice how we used `.drop_duplicates()` instead of manually creating index lists and using `.drop()`. This is simpler and more readable!

---

## 6. Outliers

📚 **Concept:** Outliers are extreme values that differ significantly from other observations. They can result from:
- **Data entry errors** ("fat-finger" errors, like typing 200 instead of 20)
- **Measurement errors**
- **Genuine extreme values** (which might be important to keep!)

⚠️ **Warning:** Many statistical methods and machine learning algorithms are sensitive to outliers. They can significantly skew results like means and linear regressions.

<!-- INSTRUCTOR: Discuss the difference between errors (should be fixed) and genuine extreme values (might be important to keep). -->

In [ ]:
# Use describe() to spot potential outliers
titanic.describe()

💡 **Tip:** Look at the `max` values and compare them to the 75th percentile (75%). If max is much larger, there may be outliers.

**What we notice:**
- `Age` maximum is 490.0 (impossible! oldest person was ~122)
- `Fare ($)` maximum is 512.33 (possible, but let's check)

<!-- INSTRUCTOR: Teach students to use domain knowledge. We know humans can't be 490 years old! -->

In [ ]:
# Find the largest ages
titanic["Age"].nlargest()

📚 **Decision framework for handling outliers:**

1. **Investigate** - Look at the outlier values and determine if they're errors
2. **Decide** - Based on domain knowledge:
   - **Delete** if it's clearly an error and you can't fix it
   - **Correct** if you can determine the true value (e.g., 490 → 49)
   - **Keep** if it's a genuine extreme value

⚠️ **Warning:** Deleting rows means losing information from other columns too! Correcting is better when possible.

<!-- INSTRUCTOR: Emphasize that outlier handling requires judgment, not just automatic rules. -->

In [ ]:
# Look at full rows for passengers with age > 90
titanic.loc[titanic["Age"] > 90]

**Our analysis:**
- Index 204: Age 102 - Unlikely but possible, we'll keep it
- Index 412: Age 490 - Obvious error, likely should be 49.0 (divide by 10)
- Index 691: Age 250 - Obvious error, likely should be 25.0 (divide by 10)

💡 **Tip:** The pattern suggests these are decimal point errors. Dividing by 10 gives plausible ages.

<!-- INSTRUCTOR: Show how domain knowledge (people don't live to 250) helps identify errors. -->

In [ ]:
# Check what value we'll get
titanic.loc[412, "Age"] / 10

In [ ]:
# Correct the error at index 412
titanic.loc[412, "Age"] = titanic.loc[412, "Age"] / 10

In [ ]:
# Correct the error at index 691
titanic.loc[691, "Age"] = titanic.loc[691, "Age"] / 10

In [ ]:
# Verify the corrections
titanic.loc[titanic["Age"] > 90]

✅ **Key Takeaway:** We successfully corrected the outliers! Now only one passenger over 90 remains, which is plausible for the Titanic era.

---

## Summary and key takeaways

🎉 **Congratulations!** You've completed a full data cleaning workflow on the Titanic dataset.

### What we accomplished:

1. ✅ **Inspected data** - Used `.info()` and `.describe()` to identify issues
2. ✅ **Fixed inconsistencies** - Removed "$" from Fare, standardized Survived column
3. ✅ **Converted types** - Changed Survived, Fare, and Age to appropriate numeric types
4. ✅ **Handled missing values** - Dropped Deck column, imputed Age (mean) and Emb (mode)
5. ✅ **Removed duplicates** - Used `.drop_duplicates()` to remove 114 duplicate rows
6. ✅ **Fixed outliers** - Corrected obvious data entry errors in Age column

### Key principles to remember:

💡 **Data cleaning is iterative** - You may need to go back and forth between steps

💡 **Use modern pandas syntax** - Prefer reassignment (`df = df.method()`) over `inplace=True`

💡 **Domain knowledge is crucial** - Understanding your data helps identify errors

💡 **Document your decisions** - Always explain why you dropped/imputed/corrected data

💡 **Preserve your raw data** - Keep the original dataset unchanged; work on copies

### Modern pandas patterns used:

```python
# ✅ Modern (preferred)
df = df.rename(columns={"old": "new"})
df = df.drop_duplicates()
df = df.reset_index(drop=True)

# ❌ Old (avoid)
df.rename(columns={"old": "new"}, inplace=True)
# Manual duplicate removal with index lists
```

### Next steps:

Now that your data is clean, you're ready to:
- Perform exploratory data analysis (EDA)
- Create visualizations
- Build statistical models or machine learning models

Remember: **Clean data is the foundation of good analysis!**

<!-- INSTRUCTOR: Emphasize that time spent cleaning data is never wasted - it prevents errors down the line. -->

---

### ✅ Final Quick Check

Can you now:
- [ ] Explain the 6-step data cleaning workflow?
- [ ] Identify data quality issues using `.info()` and `.describe()`?
- [ ] Choose appropriate strategies for handling missing values?
- [ ] Identify and correct outliers using domain knowledge?
- [ ] Use modern pandas syntax instead of `inplace=True`?

🚀 **Great job!** You're now ready for the lab exercises to practice these skills!

---

## Additional resources

📚 **Further reading:**
- [Pandas documentation: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Pandas documentation: Duplicate Labels](https://pandas.pydata.org/docs/user_guide/duplicates.html)
- [Pandas documentation: astype()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)

<!-- INSTRUCTOR: Encourage students to explore the pandas documentation for more advanced techniques. -->